# 47 · Single-session raw-event inference

Connect versioned JSONL events to the audited adapter and a verified frozen evaluation bundle.
The example is synthetic. It cannot establish clinical accuracy or deployment readiness.
No training or final-test observations are part of this notebook.

## Locate the project and set explicit run options

**What:** Resolve the checkout and choose setup, export, and demo flags.

**Why:** Report execution must not publish bundles or restore models without operator intent.
Use `SLEEP_CLASSIFIER_ROOT` for another checkout.

In [ ]:
import json
import logging
import os
import sys
from pathlib import Path

ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd())).expanduser().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
RUN_SETUP = False
RUN_EXPORT = False
RUN_DEMO = False
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s %(message)s")
logger = logging.getLogger("raw_session_notebook")
logger.info("Project root: %s", ROOT)

## Optional environment setup

**What:** Clone this branch when the checkout is absent, then install the locked runtime groups.

**Why:** Use the same commands locally or in another notebook environment.
Review the Astral installer before enabling setup.

In [ ]:
if RUN_SETUP:
    if not (ROOT / "pyproject.toml").is_file():
        repository = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
        !git clone --branch feature/raw-session-runner "{repository}" "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    !curl -LsSf https://astral.sh/uv/install.sh -o /tmp/sleep-uv-install.sh
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("uv installer download failed")
    !sh /tmp/sleep-uv-install.sh
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("uv installation failed")
    os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")

## Inspect the selected paths and protocol

**What:** Load the YAML settings and validate the synthetic input records.

**Why:** Expose the exact bundle destination and resource limits before model loading.
This cell does not export or restore a model.

In [ ]:
sys.path.insert(0, str(ROOT / "src"))
from sleep_classifier.bundle_export import EvaluationBundleExporter
from sleep_classifier.raw_session_cli import SessionCLIConfig
from sleep_classifier.session_protocol import validate_event, validate_output

exporter = EvaluationBundleExporter(ROOT / "configs/raw-session-bundle.yaml")
settings = SessionCLIConfig.load(ROOT / "configs/raw-session.yaml")
example = ROOT / "examples/raw-session/session.jsonl"
responses = ROOT / "artifacts/raw-session-demo/responses.jsonl"
if settings.bundle_dir.resolve() != exporter.output_dir:
    raise ValueError("Export and session configurations select different bundles")
events = [json.loads(line) for line in example.read_text().splitlines()]
for event in events:
    validate_event(event)
logger.info("Bundle: %s; pending events: %s; epochs/event: %s; line bytes: %s",
            settings.bundle_dir, settings.max_buffered_events,
            settings.max_epochs_per_event, settings.max_line_bytes)
logger.info("Validated %s synthetic input records; model not loaded", len(events))

## Export the fixed evaluation bundle

**What:** Publish a fresh fold-01 evaluation bundle after its mandatory parity check.

**Why:** Loading requires the exact committed runtime revision and validated device.
An old notebook-43 bundle cannot substitute for a fresh export at this revision.
If the destination exists, keep it and inspect it. Use a new identifier for an intentional export.

In [ ]:
if RUN_EXPORT:
    export_config = exporter.config_path
    logger.info("Export config: %s; output: %s", export_config, exporter.output_dir)
    if exporter.output_dir.exists():
        logger.info("Bundle already exists; preserve it and verify loading during the demo")
    else:
        exporter.plan()
        %cd "{ROOT}"
        !uv run --locked --group gru python -u -m sleep_classifier.bundle_export \
            --config "{export_config}" --export
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Bundle export failed; inspect retained evidence")
else:
    logger.info("Export skipped: enable RUN_EXPORT when ready")

## Run the synthetic session

**What:** Send the JSONL example through the CLI and save its responses.

**Why:** Check the complete input-to-prediction path without training or reading new observations.
The CLI emits two predictions, one coverage skip, and a final integrity summary.
Existing output is preserved. Remove or relocate it explicitly before an intentional repeat.

In [ ]:
if RUN_DEMO:
    if responses.exists():
        logger.info("Demo output already exists; inspect it below")
    else:
        if not settings.bundle_dir.is_dir():
            raise FileNotFoundError("Export the configured bundle before running the demo")
        responses.parent.mkdir(parents=True, exist_ok=True)
        session_config = ROOT / "configs/raw-session.yaml"
        %cd "{ROOT}"
        !uv run --locked --group gru python -u \
            -m sleep_classifier.raw_session_cli --config "{session_config}" \
            --input "{example}" > "{responses}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Demo failed; retain partial output and inspect the CLI error")
else:
    logger.info("Demo skipped: enable RUN_DEMO when ready")

## Inspect responses and completion

**What:** Validate cached responses and display their completion summary.

**Why:** Partial output cannot establish a successful session.
The probabilities describe synthetic inputs. They are not performance metrics.

In [ ]:
if responses.exists():
    records = [json.loads(line) for line in responses.read_text().splitlines()]
    for record in records:
        validate_output(record)
    if not records or records[-1]["type"] != "session_finished":
        raise ValueError("Cached demo did not finish; inspect the retained CLI error")
    summary = records[-1]
    counts = tuple(summary[key] for key in ("predicted_epochs", "skipped_epochs", "emitted_epochs"))
    if counts != (2, 1, 3):
        raise ValueError("Unexpected synthetic demo epoch counts")
    display(records)
else:
    logger.info("No cached demo output; operator execution remains pending")

## Save a fresh HTML report

**What:** Execute report-only cells in a fresh kernel.

**Why:** Preserve the readable report without recursively exporting or running inference.

In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/47_raw_session_inference.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("HTML report execution failed")